# Week 8 Post-Class Exercise: CIFAR-10

## What you'll do
Apply the CNN you built in class to CIFAR-10 — a harder dataset with **color images**. You'll see what changes when you go from 28×28 grayscale to 32×32 RGB.

## Why CIFAR-10
- **Color** instead of grayscale — your input shape changes
- **More visual variety** — same 10 classes are harder to distinguish than Fashion-MNIST
- **Standard benchmark** — you'll see this dataset again in your career

## Time required
30-60 minutes (depending on your compute). With GPU: ~30 min. Without: ~60 min.

## Prerequisites
- You completed `week8_live_session.ipynb`
- You understand the Conv2D + MaxPooling2D pattern

## Reflection (at end)
You'll be asked to write 2-3 sentences comparing your CIFAR-10 results to your Fashion-MNIST results. Save those for `Week8_Self_Assessment.md`.

## Setup

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

## Step 1: Load CIFAR-10

Note the differences from Fashion-MNIST.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten()  # cifar10 returns (N, 1); flatten for sparse_categorical
y_test = y_test.flatten()

cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Training set: {X_train.shape}')
print(f'Pixel range: [{X_train.min()}, {X_train.max()}]')
print(f'Note: shape is (N, 32, 32, 3) — already has channel dimension!')
print(f'      The 3 is for RGB. Fashion-MNIST was (N, 28, 28, 1).')

In [ ]:
# Visual sanity check
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i])
    ax.set_title(cifar_classes[y_train[i]])
    ax.axis('off')
plt.tight_layout(); plt.show()

## Step 2: Preprocess

Normalize to [0, 1]. CIFAR-10 already has the channel dimension, so no `expand_dims` needed.

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f'After preprocessing:')
print(f'  Train: {X_train.shape}, range [{X_train.min():.2f}, {X_train.max():.2f}]')
print(f'  Test:  {X_test.shape}')

## Step 3: Build a CNN for CIFAR-10

**Your task:** modify the live-session CNN architecture for the new input shape.

Key differences:
- Input shape is `(32, 32, 3)` not `(28, 28, 1)`
- This problem is harder — consider a slightly bigger network

Try this architecture as a starting point:
```
Conv2D(32, 3) → Conv2D(32, 3) → MaxPool
Conv2D(64, 3) → Conv2D(64, 3) → MaxPool
Flatten → Dense(128) → Dropout(0.5) → Dense(10)
```

Note the doubled-up conv layers ("VGG-style") — two 3×3 convs before pooling get a similar receptive field to one 5×5 with fewer parameters.

In [ ]:
# YOUR TURN: build the model
# Hint: use keras.Sequential([...]) like in the live session

model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),  # Note: 3 channels (RGB)
    # TODO: Conv2D(32, 3, activation='relu', padding='same'),
    # TODO: Conv2D(32, 3, activation='relu', padding='same'),
    # TODO: MaxPooling2D(2),
    # TODO: Conv2D(64, 3, activation='relu', padding='same'),
    # TODO: Conv2D(64, 3, activation='relu', padding='same'),
    # TODO: MaxPooling2D(2),
    # TODO: Flatten(),
    # TODO: Dense(128, activation='relu'),
    # TODO: Dropout(0.5),
    # TODO: Dense(10, activation='softmax'),
])

# model.summary()  # Uncomment when ready

## Step 4: Compile and train

Use EarlyStopping (Week 7 callback) so the model stops if it plateaus.

In [ ]:
# YOUR TURN: compile the model
# model.compile(
#     optimizer='adam',
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# YOUR TURN: train with EarlyStopping
# callbacks = [
#     keras.callbacks.EarlyStopping(
#         monitor='val_loss', patience=4, restore_best_weights=True, verbose=1
#     )
# ]

# history = model.fit(
#     X_train, y_train,
#     epochs=20,
#     batch_size=128,
#     validation_split=0.1,
#     callbacks=callbacks,
#     verbose=2
# )

## Step 5: Evaluate

In [ ]:
# YOUR TURN: evaluate on test set
# test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
# print(f'Test accuracy: {test_acc:.4f}')

# Expected: 70-78% accuracy with this baseline architecture
# (CIFAR-10 is genuinely harder than Fashion-MNIST — that's expected)

## Step 6: Plot training curves

Look for the patterns from Week 7: training loss decreasing, validation loss flattening or rising = overfitting.

In [ ]:
# YOUR TURN: plot accuracy and loss curves
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].plot(history.history['accuracy'], label='train')
# axes[0].plot(history.history['val_accuracy'], label='val')
# axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
# axes[1].plot(history.history['loss'], label='train')
# axes[1].plot(history.history['val_loss'], label='val')
# axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
# plt.tight_layout(); plt.show()

## Step 7: Visualize misclassifications

Same pattern as the live session — what's the model getting wrong on CIFAR-10?

In [ ]:
# YOUR TURN: predict on test set, find misclassifications, plot a few
# predictions = model.predict(X_test, verbose=0)
# predicted_labels = np.argmax(predictions, axis=1)
# wrong = np.where(predicted_labels != y_test)[0]
# fig, axes = plt.subplots(1, 8, figsize=(16, 2))
# for i, ax in enumerate(axes):
#     idx = wrong[i]
#     ax.imshow(X_test[idx])
#     ax.set_title(f'P:{cifar_classes[predicted_labels[idx]]}\nA:{cifar_classes[y_test[idx]]}', fontsize=8)
#     ax.axis('off')
# plt.tight_layout(); plt.show()

## Reflection (write your answers in `Week8_Self_Assessment.md`)

1. **What test accuracy did you achieve?** Compare to your Fashion-MNIST CNN.
2. **CIFAR-10 is harder than Fashion-MNIST.** Why? Think about: visual variety, intra-class diversity, color information adding noise, intra-class similarity (cat vs dog).
3. **Did your model overfit?** Look at the training curves. Was validation loss still decreasing when training stopped, or had it plateaued?
4. **What single architecture change** do you think would most improve CIFAR-10 accuracy? (You're not required to test it — just hypothesize.)
5. **Foreshadow:** If you had only 200 CIFAR-10 images instead of 50,000, how do you think this network would do? (Answer: badly. That's why Week 9 introduces transfer learning.)

## Looking ahead to Week 9

You just trained a CNN from scratch on 50,000 images and got ~75% accuracy. Next week you'll see how to use a model SOMEONE ELSE trained on 14 million images, adapt it to your data, and get 95%+ accuracy with only a few hundred examples. That's transfer learning.